Сгенерировать последовательности, которые состоят из цифр (от 0 до 9) и задаются следующим образом:\
x - последовательность цифр\
y1 = x1 \
yi = xi + x1 \
Если yi >= 10 то yi = yi - 10 \
Научить модель рекуррентной нейронной сети предсказывать yi по xi Использовать: RNN, LSTM, GRU 

In [1]:
import pandas as pd
import numpy as np
import time
import torch

In [2]:
def generate_series(size = 30):
    """
        генерирует две связанные последовательности и возвращает их кортежем
    """
    x_series = [np.random.randint(0, 10) for _ in range(0, size)]
    y_series = [x_series[0]]
    for x in x_series[1:]:
        y = x + x_series[0]
        if (y >= 10):
            y = y - 10
        y_series.append(y) 
    return (x_series, y_series)   

In [3]:
# Сгенерируем последовательности x и y
series_size = 10
x_series, y_series = generate_series(series_size)

In [4]:
x_series

[5, 8, 8, 8, 1, 3, 4, 0, 3, 9]

In [5]:
y_series

[5, 3, 3, 3, 6, 8, 9, 5, 8, 4]

### Создадим датасет для работы:

In [6]:
MAX_LEN = 15 # берем последовательности максимум до 15 символов
SAMPLES_SIZE = 2000
X_SAMPLES = []
Y_SAMPLES = []

# Cформируем множество сэмплов
for i in range(0, SAMPLES_SIZE):
    x_series, y_series = generate_series(MAX_LEN)
    X_SAMPLES.append(x_series)  # Формируем X
    Y_SAMPLES.append(y_series)  # Формируем Y

print("Количество множества сэмплов: ", len(X_SAMPLES))

Количество множества сэмплов:  2000


Разделим сэмплы на тренировочный и тестовый наборы

In [7]:
# разделяем 90% на обучение, 10% на тест
split_idx = int(len(X_SAMPLES) * 0.9)

train_x_samples = X_SAMPLES[:split_idx]
test_x_samples = X_SAMPLES[split_idx:]

train_y_samples = Y_SAMPLES[:split_idx]
test_y_samples = Y_SAMPLES[split_idx:]

In [8]:
# Необходимо теперь нам это все превратить в тензоры для того чтобы подать в алгоритм рекуррентной нейронной сети
X = torch.zeros((len(train_x_samples), MAX_LEN), dtype=int)
Y = torch.zeros((len(train_y_samples), MAX_LEN), dtype=int)

for i in range(0, len(train_x_samples)):
    for t, digit in enumerate(train_x_samples[i]):
        X[i, t] = digit
    for t, digit in enumerate(train_y_samples[i]):
        Y[i, t] = digit        

### Cформируем датасет, который будем передавать в нейронную сеть

In [9]:
BATCH_SIZE = 512
dataset = torch.utils.data.TensorDataset(X, Y)
data = torch.utils.data.DataLoader(dataset, BATCH_SIZE, shuffle=True)

In [10]:
# Строим класс RnnFlex, который будет принимать GRU, LSTM или RNN
class RnnFlex(torch.nn.Module):
                        # тип     размер словаря  размер эмб       скрытые слои   классы
    def __init__(self, rnnClass, dictionary_size, embedding_size, num_hiddens, num_classes):
        super().__init__()
        self.embedding = torch.nn.Embedding(dictionary_size, embedding_size)
        self.hidden = rnnClass(embedding_size, num_hiddens, batch_first=True)
        self.output = torch.nn.Linear(num_hiddens, num_classes)
        self.is_lstm = issubclass(rnnClass, torch.nn.LSTM)  # Проверка типа RNN

    def forward(self, X):
        out = self.embedding(X)
        if self.is_lstm: # приходят все выходы и последний выход (между LSTM  и GRU выход немного разный)
            o, (state, _) = self.hidden(out)  # Извлекаем только h из (h, c)
        else:
            o, state = self.hidden(out)  # Для GRU или SimpleRNN h сразу возвращается
        predictions = self.output(o)  # Используем все скрытые состояния
        return predictions

In [11]:
dictionary_size = 10
num_classes = 10

In [12]:
def sample(preds):
    softmaxed = torch.softmax(preds, 0)
    probas = torch.distributions.multinomial.Multinomial(1, softmaxed).sample()
    return probas.argmax()

def predict_series(model_to_estimate):
    # Берем случайный индекс множества сэмплов
    sample_index = np.random.randint(0, len(train_x_samples))

    x_sample = train_x_samples[sample_index]
    y_sample = train_y_samples[sample_index]

    predict_series1(model_to_estimate, x_sample, y_sample)

def predict_series1(model_to_estimate, x_sample, y_sample, to_print = True):
    x_input = torch.zeros((1, MAX_LEN), dtype=int)
    for t, digit in enumerate(x_sample):
        x_input[0, t] = digit
    
    answers = model_to_estimate(x_input)
    preds = answers[0]
    y_output = []
    for i in range(MAX_LEN):
        y_output.append(sample(preds[i]).item())

    y_sample_styled, y_output_styled = [], []
    for s, y in zip(y_sample, y_output):
        if s != y:
            # Если не равны — оборачиваем в жирный красный шрифт (\033[1m)
            y_sample_styled.append(f"\033[1;31m{s}\033[0m")
            y_output_styled.append(f"\033[1;31m{y}\033[0m")
        else:
            y_sample_styled.append(str(s))
            y_output_styled.append(str(y))
    if (to_print):        
        print("expected: ", ", ".join(y_sample_styled))
        print("actual:   ", ", ".join(y_output_styled))

    # определяем, равны ли исходная и предсказанная последовательности
    return y_sample == y_output   

In [13]:
def estimate_accuracy(model_to_estimate, orig_samples, expected_samples):
    """
        Определяет accuracy предсказанных сэмплов
    """ 
    mask = [predict_series1(model_to_estimate, orig, exp, to_print = False) for orig, exp in zip(orig_samples, expected_samples)]
    return np.mean(mask)  

## RNN

In [14]:
model = RnnFlex(torch.nn.RNN, dictionary_size, 32, 128, num_classes)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

for ep in range(201):
    start = time.time()
    train_loss = 0.
    train_passed = 0

    model.train()
    for X_b, y_b in data:
        optimizer.zero_grad()
        answers = model(X_b)

        answers = answers.view(-1, num_classes)
        
        y_b = y_b.flatten()
        
        loss = criterion(answers, y_b)
        train_loss += loss.item()

        loss.backward()
        optimizer.step()
        train_passed += 1

    if (ep % 50 == 0):
        print("Epoch {}   Time {:.3f}   Train Loss: {:.3f}".format(ep, time.time() - start, train_loss / train_passed))
        model.eval()
        predict_series(model)
        print("--------------------------------------------------------")

Epoch 0   Time 0.542   Train Loss: 2.312
expected:  4, 2, 2, 4, 3, 0, 6, 5, 5, 1, 6, 6, 0, 4, 2
actual:    8, 9, 8, 7, 3, 0, 9, 2, 1, 1, 4, 1, 5, 1, 3
--------------------------------------------------------
Epoch 50   Time 0.324   Train Loss: 1.888
expected:  3, 9, 4, 8, 8, 8, 5, 5, 3, 7, 2, 4, 5, 3, 3
actual:    3, 8, 0, 6, 1, 1, 8, 9, 4, 9, 3, 7, 4, 6, 1
--------------------------------------------------------
Epoch 100   Time 0.383   Train Loss: 1.005
expected:  1, 3, 7, 7, 4, 2, 3, 4, 4, 0, 5, 1, 4, 4, 3
actual:    5, 3, 1, 9, 8, 2, 1, 4, 4, 7, 3, 1, 0, 6, 3
--------------------------------------------------------
Epoch 150   Time 0.346   Train Loss: 0.195
expected:  5, 2, 0, 3, 3, 8, 4, 6, 1, 1, 4, 3, 3, 9, 5
actual:    5, 2, 0, 3, 3, 8, 4, 6, 4, 1, 4, 3, 3, 9, 5
--------------------------------------------------------
Epoch 200   Time 0.484   Train Loss: 0.056
expected:  9, 4, 5, 4, 9, 7, 4, 4, 6, 7, 3, 5, 0, 5, 0
actual:    9, 4, 5, 4, 9, 7, 4, 4, 8, 7, 3, 5, 0, 5, 0
----------

In [15]:
print("Оценка Accuracy для RNN:")
rnn_train_accuracy = estimate_accuracy(model, train_x_samples, train_y_samples)
print(f"Accuracy для тренировочных данных: {rnn_train_accuracy:.2f}")
rnn_test_accuracy = estimate_accuracy(model, test_x_samples, test_y_samples)
print(f"Accuracy для тестовых данных: {rnn_test_accuracy:.2f}")

Оценка Accuracy для RNN:
Accuracy для тренировочных данных: 0.48
Accuracy для тестовых данных: 0.43


## GRU

In [16]:
model = RnnFlex(torch.nn.GRU, dictionary_size, 32, 128, num_classes)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

for ep in range(201):
    start = time.time()
    train_loss = 0.
    train_passed = 0

    model.train()
    for X_b, y_b in data:
        optimizer.zero_grad()
        answers = model(X_b)

        answers = answers.view(-1, num_classes)
        
        y_b = y_b.flatten()
        
        loss = criterion(answers, y_b)
        train_loss += loss.item()

        loss.backward()
        optimizer.step()
        train_passed += 1

    if (ep % 50 == 0):
        print("Epoch {}   Time {:.3f}   Train Loss: {:.3f}".format(ep, time.time() - start, train_loss / train_passed))
        model.eval()
        predict_series(model)
        print("--------------------------------------------------------")

Epoch 0   Time 1.392   Train Loss: 2.303
expected:  4, 5, 8, 6, 9, 0, 5, 3, 0, 6, 5, 4, 1, 5, 4
actual:    9, 8, 3, 6, 4, 7, 2, 4, 0, 8, 5, 2, 9, 3, 5
--------------------------------------------------------
Epoch 50   Time 0.810   Train Loss: 0.257
expected:  7, 4, 7, 6, 8, 3, 9, 2, 6, 1, 1, 0, 1, 8, 7
actual:    7, 7, 7, 6, 8, 3, 9, 2, 6, 1, 1, 8, 1, 8, 7
--------------------------------------------------------
Epoch 100   Time 0.823   Train Loss: 0.022
expected:  5, 2, 2, 4, 7, 4, 1, 3, 3, 1, 3, 9, 5, 1, 6
actual:    5, 2, 2, 4, 7, 4, 6, 3, 3, 1, 3, 9, 5, 1, 6
--------------------------------------------------------
Epoch 150   Time 0.962   Train Loss: 0.008
expected:  9, 7, 3, 9, 3, 1, 8, 2, 3, 5, 0, 8, 6, 1, 5
actual:    9, 7, 3, 9, 3, 1, 8, 2, 3, 5, 0, 8, 6, 1, 5
--------------------------------------------------------
Epoch 200   Time 0.743   Train Loss: 0.005
expected:  2, 9, 8, 8, 7, 4, 5, 1, 4, 4, 7, 2, 1, 8, 6
actual:    2, 9, 8, 8, 7, 4, 5, 1, 4, 4, 7, 2, 1, 8, 6
----------

In [17]:
print("Оценка Accuracy для GRU:")
rnn_train_accuracy = estimate_accuracy(model, train_x_samples, train_y_samples)
print(f"Accuracy для тренировочных данных: {rnn_train_accuracy:.2f}")
rnn_test_accuracy = estimate_accuracy(model, test_x_samples, test_y_samples)
print(f"Accuracy для тестовых данных: {rnn_test_accuracy:.2f}")

Оценка Accuracy для GRU:
Accuracy для тренировочных данных: 0.94
Accuracy для тестовых данных: 0.94


## LSTM

In [18]:
model = RnnFlex(torch.nn.LSTM, dictionary_size, 32, 128, num_classes)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

for ep in range(201):
    start = time.time()
    train_loss = 0.
    train_passed = 0

    model.train()
    for X_b, y_b in data:
        optimizer.zero_grad()
        answers = model(X_b)

        answers = answers.view(-1, num_classes)
        
        y_b = y_b.flatten()
        
        loss = criterion(answers, y_b)
        train_loss += loss.item()

        loss.backward()
        optimizer.step()
        train_passed += 1

    if (ep % 50 == 0):
        print("Epoch {}   Time {:.3f}   Train Loss: {:.3f}".format(ep, time.time() - start, train_loss / train_passed))
        model.eval()
        predict_series(model)
        print("--------------------------------------------------------")

Epoch 0   Time 0.685   Train Loss: 2.302
expected:  3, 6, 5, 9, 4, 2, 8, 4, 7, 7, 4, 2, 3, 9, 0
actual:    8, 4, 2, 3, 9, 6, 7, 0, 5, 0, 5, 4, 4, 3, 3
--------------------------------------------------------
Epoch 50   Time 0.635   Train Loss: 0.168
expected:  8, 9, 2, 0, 2, 1, 0, 1, 9, 0, 0, 0, 6, 7, 3
actual:    8, 1, 2, 0, 2, 1, 0, 1, 9, 0, 0, 0, 6, 5, 3
--------------------------------------------------------
Epoch 100   Time 0.624   Train Loss: 0.017
expected:  4, 8, 1, 0, 4, 6, 4, 1, 9, 5, 0, 5, 3, 1, 4
actual:    4, 8, 1, 0, 4, 6, 4, 1, 9, 5, 0, 5, 3, 1, 4
--------------------------------------------------------
Epoch 150   Time 0.529   Train Loss: 0.007
expected:  9, 7, 6, 4, 2, 2, 7, 3, 4, 5, 8, 0, 9, 4, 3
actual:    9, 7, 6, 4, 2, 2, 7, 3, 4, 5, 8, 0, 9, 4, 3
--------------------------------------------------------
Epoch 200   Time 0.554   Train Loss: 0.004
expected:  7, 3, 3, 8, 5, 0, 0, 8, 4, 4, 5, 1, 1, 3, 6
actual:    7, 3, 3, 8, 5, 0, 0, 8, 4, 4, 5, 1, 1, 3, 6
----------

In [19]:
print("Оценка Accuracy для LSTM:")
rnn_train_accuracy = estimate_accuracy(model, train_x_samples, train_y_samples)
print(f"Accuracy для тренировочных данных: {rnn_train_accuracy:.2f}")
rnn_test_accuracy = estimate_accuracy(model, test_x_samples, test_y_samples)
print(f"Accuracy для тестовых данных: {rnn_test_accuracy:.2f}")

Оценка Accuracy для LSTM:
Accuracy для тренировочных данных: 0.94
Accuracy для тестовых данных: 0.93
